# 15 Kaggle Pairwise TF-IDF Submission

Independent Kaggle submission notebook for the pairwise TF-IDF Logistic Regression model. It reads only official Kaggle input files and writes `/kaggle/working/submission.csv`.

## 1. Imports and Kaggle path discovery

Find the official competition files under `/kaggle/input` and define constants used throughout the notebook.

In [ ]:
import ast
import gc
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
LABELS = [0, 1, 2]
INPUT_ROOT = Path("/kaggle/input")
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")

train_candidates = list(INPUT_ROOT.rglob("train.csv"))
test_candidates = list(INPUT_ROOT.rglob("test.csv"))
sample_candidates = list(INPUT_ROOT.rglob("sample_submission.csv"))
if not train_candidates:
    raise FileNotFoundError("No train.csv found under /kaggle/input")
if not test_candidates:
    raise FileNotFoundError("No test.csv found under /kaggle/input")
if not sample_candidates:
    raise FileNotFoundError("No sample_submission.csv found under /kaggle/input")

train_path = train_candidates[0]
test_path = test_candidates[0]
sample_submission_path = sample_candidates[0]
print("train_path:", train_path)
print("test_path:", test_path)
print("sample_submission_path:", sample_submission_path)

## 2. Read raw Kaggle data

Load the raw train, test, and sample submission files. Model identity columns are not used as features.

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)
print("train columns:", train.columns.tolist())
print("test columns:", test.columns.tolist())

required_train_columns = {
    "id", "prompt", "response_a", "response_b",
    "winner_model_a", "winner_model_b", "winner_tie",
}
required_test_columns = {"id", "prompt", "response_a", "response_b"}
missing_train = sorted(required_train_columns - set(train.columns))
missing_test = sorted(required_test_columns - set(test.columns))
if missing_train:
    raise ValueError(f"train.csv missing required columns: {missing_train}")
if missing_test:
    raise ValueError(f"test.csv missing required columns: {missing_test}")
if train["id"].duplicated().any():
    raise ValueError("Duplicate ids found in train.csv")
if test["id"].duplicated().any():
    raise ValueError("Duplicate ids found in test.csv")

## 3. Text parsing and cleaning

Parse possible JSON/list-like strings, normalize line breaks, remove invalid surrogate Unicode, and create clean text columns.

In [ ]:
def clean_unicode(text):
    if text is None:
        return ""
    if not isinstance(text, str):
        text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u2028", "\n").replace("\u2029", "\n")
    try:
        text = text.encode("utf-16", "surrogatepass").decode("utf-16", "replace")
    except UnicodeError:
        text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    return text

def parse_text_field(x):
    if pd.isna(x):
        return ""
    if not isinstance(x, str):
        parsed = x
    else:
        text = clean_unicode(x).strip()
        parsed = None
        try:
            parsed = json.loads(text)
        except Exception:
            try:
                parsed = ast.literal_eval(text)
            except Exception:
                parsed = text
    if isinstance(parsed, list):
        return "\n".join(str(item) for item in parsed if item is not None)
    if parsed is None:
        return ""
    return str(parsed)

def normalize_text(x):
    text = parse_text_field(x)
    text = clean_unicode(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u2028", "\n").replace("\u2029", "\n")
    return text.strip()

for df_name, df in [("train", train), ("test", test)]:
    for source_column in ["prompt", "response_a", "response_b"]:
        clean_column = f"{source_column}_clean"
        df[clean_column] = df[source_column].map(normalize_text)
    print(f"{df_name} clean text columns created")

## 4. Label validation and conversion

Validate that exactly one winner column is active for every training row, then map A/B/tie to labels 0/1/2.

In [ ]:
winner_columns = ["winner_model_a", "winner_model_b", "winner_tie"]
winner_values = train[winner_columns].apply(pd.to_numeric, errors="raise").astype(int)
if not winner_values.isin([0, 1]).all().all():
    raise ValueError("Winner columns must contain only 0 and 1")
if not winner_values.sum(axis=1).eq(1).all():
    bad_count = int((~winner_values.sum(axis=1).eq(1)).sum())
    raise ValueError(f"Found {bad_count} rows where winner columns do not sum to exactly 1")

train["label"] = np.select(
    [winner_values["winner_model_a"].eq(1), winner_values["winner_model_b"].eq(1), winner_values["winner_tie"].eq(1)],
    [0, 1, 2],
).astype(np.int64)
print("train label distribution:")
print(train["label"].value_counts().sort_index())

## 5. Pairwise TF-IDF and numeric feature functions

Define the same shared response TF-IDF representation and numeric difference features used by the local pairwise experiment.

In [ ]:
REFUSAL_TERMS = [
    "cannot", "can't", "unable", "won't", "sorry", "not able",
    "cannot assist", "cannot provide",
]
NUMBERED_LIST_PATTERN = re.compile(r"(?m)^\s*\d+\.\s+")

def sparse_memory_mb(matrix):
    total_bytes = matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
    return total_bytes / (1024 ** 2)

def count_list_markers(text):
    return text.count("\n- ") + text.count("\n* ") + len(NUMBERED_LIST_PATTERN.findall(text))

def count_refusal_terms(text):
    lowered = text.lower()
    return sum(lowered.count(term) for term in REFUSAL_TERMS)

def signed_log1p(values):
    values = np.asarray(values, dtype=np.float32)
    return np.sign(values) * np.log1p(np.abs(values))

def build_numeric_features(df):
    response_a = df["response_a_clean"].fillna("").astype(str)
    response_b = df["response_b_clean"].fillna("").astype(str)

    char_diff = response_a.str.len().to_numpy() - response_b.str.len().to_numpy()
    word_diff = response_a.str.split().str.len().to_numpy() - response_b.str.split().str.len().to_numpy()
    newline_diff = response_a.str.count("\n").to_numpy() - response_b.str.count("\n").to_numpy()
    list_diff = response_a.map(count_list_markers).to_numpy() - response_b.map(count_list_markers).to_numpy()
    code_diff = response_a.str.count("```").to_numpy() - response_b.str.count("```").to_numpy()
    refusal_diff = response_a.map(count_refusal_terms).to_numpy() - response_b.map(count_refusal_terms).to_numpy()
    question_diff = response_a.str.count(r"\?").to_numpy() - response_b.str.count(r"\?").to_numpy()
    exclamation_diff = response_a.str.count("!").to_numpy() - response_b.str.count("!").to_numpy()

    return np.column_stack([
        signed_log1p(char_diff),
        np.log1p(np.abs(char_diff)),
        signed_log1p(word_diff),
        np.log1p(np.abs(word_diff)),
        signed_log1p(newline_diff),
        signed_log1p(list_diff),
        signed_log1p(code_diff),
        signed_log1p(refusal_diff),
        signed_log1p(question_diff),
        signed_log1p(exclamation_diff),
    ]).astype(np.float32)

def build_pairwise_components(df, fitted_prompt_vectorizer, fitted_response_vectorizer):
    x_prompt = fitted_prompt_vectorizer.transform(df["prompt_clean"])
    x_a = fitted_response_vectorizer.transform(df["response_a_clean"])
    x_b = fitted_response_vectorizer.transform(df["response_b_clean"])
    x_diff = x_a - x_b
    x_abs_diff = x_diff.copy()
    x_abs_diff.data = np.abs(x_abs_diff.data)
    x_sum = (x_a + x_b).multiply(np.float32(0.5))
    del x_a, x_b
    gc.collect()
    return x_prompt, x_diff, x_abs_diff, x_sum

def stack_components(x_prompt, x_diff, x_abs_diff, x_sum, x_numeric):
    return hstack([x_prompt, x_diff, x_abs_diff, x_sum, x_numeric], format="csr", dtype=np.float32)

def make_swapped_matrix(x_prompt, x_diff, x_abs_diff, x_sum, numeric_scaled):
    swapped_numeric = numeric_scaled.copy()
    directional_columns = [0, 2, 4, 5, 6, 7, 8, 9]
    swapped_numeric[:, directional_columns] *= -1.0
    x_swapped_numeric = csr_matrix(swapped_numeric, dtype=np.float32)
    return stack_components(x_prompt, -x_diff, x_abs_diff, x_sum, x_swapped_numeric)

def swap_labels(y):
    return np.asarray(pd.Series(y).map({0: 1, 1: 0, 2: 2}), dtype=np.int64)

## 6. Fit vectorizers on full training data

Fit prompt TF-IDF on full training prompts and the shared response TF-IDF on full training Response A plus Response B text only.

In [ ]:
prompt_vectorizer = TfidfVectorizer(
    analyzer="word", max_features=20000, ngram_range=(1, 2),
    min_df=2, max_df=0.95, sublinear_tf=True,
    strip_accents="unicode", dtype=np.float32,
)
response_vectorizer = TfidfVectorizer(
    analyzer="word", max_features=60000, ngram_range=(1, 2),
    min_df=2, max_df=0.95, sublinear_tf=True,
    strip_accents="unicode", dtype=np.float32,
)

prompt_vectorizer.fit(train["prompt_clean"])
response_fit_text = pd.concat([train["response_a_clean"], train["response_b_clean"]], ignore_index=True)
response_vectorizer.fit(response_fit_text)
del response_fit_text
gc.collect()
print("Prompt vocabulary size:", len(prompt_vectorizer.vocabulary_))
print("Response vocabulary size:", len(response_vectorizer.vocabulary_))

## 7. Build full train matrices with A/B swap augmentation

Create original and algebraically swapped pairwise matrices without repeating expensive TF-IDF transforms.

In [ ]:
x_prompt, x_diff, x_abs_diff, x_sum = build_pairwise_components(train, prompt_vectorizer, response_vectorizer)
train_numeric_raw = build_numeric_features(train)
numeric_scaler = StandardScaler()
train_numeric_scaled = numeric_scaler.fit_transform(train_numeric_raw).astype(np.float32)
x_train_numeric = csr_matrix(train_numeric_scaled, dtype=np.float32)

x_original = stack_components(x_prompt, x_diff, x_abs_diff, x_sum, x_train_numeric)
x_swapped = make_swapped_matrix(x_prompt, x_diff, x_abs_diff, x_sum, train_numeric_scaled)
y_original = train["label"].to_numpy(dtype=np.int64)
y_swapped = swap_labels(y_original)

x_train_aug = vstack([x_original, x_swapped], format="csr", dtype=np.float32)
y_train_aug = np.concatenate([y_original, y_swapped])

print("X_original shape:", x_original.shape)
print("X_train_aug shape:", x_train_aug.shape)
print("X_train_aug memory MB:", f"{sparse_memory_mb(x_train_aug):.2f}")

del x_original, x_swapped, x_prompt, x_diff, x_abs_diff, x_sum, x_train_numeric
del train_numeric_raw, train_numeric_scaled
gc.collect()

## 8. Train Logistic Regression

Train the best local pairwise model setting on the full augmented training set.

In [ ]:
model = LogisticRegression(
    C=0.1,
    solver="saga",
    max_iter=800,
    tol=1e-3,
    n_jobs=-1,
    random_state=42,
)
model.fit(x_train_aug, y_train_aug)
print("Model training finished")
del x_train_aug, y_train_aug
gc.collect()

## 9. Build test matrices and predict with swap averaging

Construct original and algebraically swapped test features, map swapped probabilities back to original A/B order, and average the two predictions.

In [ ]:
t_prompt, t_diff, t_abs_diff, t_sum = build_pairwise_components(test, prompt_vectorizer, response_vectorizer)
test_numeric_raw = build_numeric_features(test)
test_numeric_scaled = numeric_scaler.transform(test_numeric_raw).astype(np.float32)
test_numeric = csr_matrix(test_numeric_scaled, dtype=np.float32)

x_test_original = stack_components(t_prompt, t_diff, t_abs_diff, t_sum, test_numeric)
x_test_swapped = make_swapped_matrix(t_prompt, t_diff, t_abs_diff, t_sum, test_numeric_scaled)
print("X_test_original shape:", x_test_original.shape)
print("X_test_swapped shape:", x_test_swapped.shape)

probs_original = model.predict_proba(x_test_original)
probs_swapped = model.predict_proba(x_test_swapped)
mapped_probs = np.column_stack([probs_swapped[:, 1], probs_swapped[:, 0], probs_swapped[:, 2]])
final_probs = 0.5 * probs_original + 0.5 * mapped_probs

if final_probs.shape != (len(test), 3):
    raise ValueError(f"Invalid final probability shape: {final_probs.shape}")
if np.isnan(final_probs).any():
    raise ValueError("Final probabilities contain NaN")
if np.isinf(final_probs).any():
    raise ValueError("Final probabilities contain Inf")
if (final_probs < 0).any():
    raise ValueError("Final probabilities contain negative values")
if not np.allclose(final_probs.sum(axis=1), 1.0, atol=1e-6):
    raise ValueError("Final probability rows do not sum to 1")

del t_prompt, t_diff, t_abs_diff, t_sum, test_numeric_raw, test_numeric_scaled, test_numeric
del x_test_original, x_test_swapped, probs_original, probs_swapped, mapped_probs
gc.collect()

## 10. Create and validate submission

Write the Kaggle submission with exactly the required columns and validate probability integrity.

In [ ]:
submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "winner_model_a": final_probs[:, 0],
    "winner_model_b": final_probs[:, 1],
    "winner_tie": final_probs[:, 2],
})
expected_columns = ["id", "winner_model_a", "winner_model_b", "winner_tie"]
probability_columns = expected_columns[1:]
if submission.columns.tolist() != expected_columns:
    raise ValueError(f"Invalid submission columns: {submission.columns.tolist()}")
if submission.shape != (len(test), 4):
    raise ValueError(f"Invalid submission shape: {submission.shape}")
if submission[probability_columns].isna().any().any():
    raise ValueError("Submission contains NaN probabilities")
if np.isinf(submission[probability_columns].to_numpy()).any():
    raise ValueError("Submission contains Inf probabilities")
if (submission[probability_columns] < 0).any().any():
    raise ValueError("Submission contains negative probabilities")
probability_sums = submission[probability_columns].sum(axis=1)
if not np.allclose(probability_sums, 1.0, atol=1e-6):
    raise ValueError("Submission probability rows do not sum to 1")

submission.to_csv(SUBMISSION_PATH, index=False)
print("train_path:", train_path)
print("test_path:", test_path)
print("train shape:", train.shape)
print("test shape:", test.shape)
print("train label distribution:")
print(train["label"].value_counts().sort_index())
print("submission shape:", submission.shape)
print("submission columns:", submission.columns.tolist())
print("probability sum statistics:")
print(probability_sums.describe())
print(submission.head())
print("Kaggle pairwise TF-IDF submission finished successfully.")